# КТ3 · Какому датчику доверять на повороте?
Колёса задают начальный маршрут. Камера, гироскоп, наблюдения пяти ориентиров и возможное повторное наблюдение места уточняют его в общей системе координат. Здесь можно прочитать реализацию, воспроизвести 32 сценария и проверить производные численно.
        
**Гипотеза.** На маршрутах с выбросами блочная потеря Хьюбера снизит медианную ATE относительно квадратичной. На чистых данных ожидаю близкие результаты. Единица сравнения — маршрут. Фиксированные seed перечислены в коде опыта.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
from threadpoolctl import threadpool_limits

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
Path("slam_lab").mkdir(exist_ok=True)
threadpool_limits(limits=1);

### Библиотека плоского SLAM
Первая поза закреплена. Относительное движение задано в осях начальной позы каждого интервала. Дальности и координаты выражены в метрах, углы в радианах. Генератор возвращает наблюдения и эталон разными объектами. Функция `solve(graph)` получает только граф измерений.

In [ ]:
%%writefile slam_lab/graph.py
"""Planar factor graph. Coordinates are metres, angles radians, gyro bias rad/s."""
from dataclasses import dataclass
import numpy as np


def wrap(a):
    return (np.asarray(a) + np.pi) % (2 * np.pi) - np.pi


def rotate(v, angle):
    x, y = np.moveaxis(np.asarray(v), -1, 0)
    c, s = np.cos(angle), np.sin(angle)
    return np.stack((c*x - s*y, s*x + c*y), axis=-1)


def relative(a, b):
    a, b = np.asarray(a), np.asarray(b)
    return np.r_[rotate(b[:2] - a[:2], -a[2]), wrap(b[2] - a[2])]


def compose(a, step):
    a, step = np.asarray(a), np.asarray(step)
    return np.r_[a[:2] + rotate(step[:2], a[2]), wrap(a[2] + step[2])]


def camera_delta_to_base(T21, T_BC):
    """Project a PnP transform Cj <- Ci to a planar motion Bi <- Bj (x, y, yaw)."""
    def rigid(value):
        M = np.asarray(value, dtype=float)
        if (M.shape != (4, 4) or not np.isfinite(M).all()
                or not np.allclose(M[3], [0, 0, 0, 1], atol=1e-7, rtol=0)):
            raise ValueError('expected a finite homogeneous transform')
        R = M[:3, :3]
        if not np.allclose(R.T @ R, np.eye(3), atol=1e-6) or np.linalg.det(R) <= 0:
            raise ValueError('expected a proper rotation')
        return M

    camera, mounting = rigid(T21), rigid(T_BC)
    step = mounting @ np.linalg.inv(camera) @ np.linalg.inv(mounting)
    if np.hypot(step[0, 0], step[1, 0]) < 1e-8:
        raise ValueError('body heading has no horizontal projection')
    return np.array([step[0, 3], step[1, 3], np.arctan2(step[1, 0], step[0, 0])])


@dataclass(frozen=True)
class Factor:
    kind: str
    i: int
    j: int
    z: tuple
    sigma: tuple
    dt: float = 1.


@dataclass(frozen=True)
class Graph:
    poses: int
    landmarks: int
    factors: tuple


SIZES = {'wheel': 3, 'vo': 3, 'loop': 3, 'gyro': 1, 'beacon': 2}
COURSE_HUBER = ('vo', 'loop', 'beacon')


def validate(graph):
    if not isinstance(graph.poses, (int, np.integer)) or graph.poses < 2:
        raise ValueError('poses must be an integer >= 2')
    if not isinstance(graph.landmarks, (int, np.integer)) or graph.landmarks < 0:
        raise ValueError('landmark count must be a nonnegative integer')
    for k, f in enumerate(graph.factors):
        limit = graph.landmarks if f.kind == 'beacon' else graph.poses
        indices = all(isinstance(v, (int, np.integer)) for v in (f.i, f.j))
        dim = SIZES.get(f.kind, 0)
        if not dim or not indices or not (0 <= f.i < graph.poses and 0 <= f.j < limit):
            raise ValueError(f'factor {k}: invalid kind or index')
        if f.kind != 'beacon' and f.i == f.j:
            raise ValueError(f'factor {k}: identical pose indices')
        z, sigma = np.asarray(f.z, float), np.asarray(f.sigma, float)
        if z.shape != (dim,) or sigma.shape != (dim,) or not np.isfinite(np.r_[z, sigma]).all() or np.any(sigma <= 0):
            raise ValueError(f'factor {k}: invalid value, dimension or sigma')
        if (f.kind == 'beacon' and z[0] <= 0) or (f.kind == 'gyro' and (not np.isfinite(f.dt) or f.dt <= 0)):
            raise ValueError(f'factor {k}: invalid range or gyro interval')


def wheel_path(graph):
    steps = {f.i: f.z for f in graph.factors if f.kind == 'wheel' and f.j == f.i + 1}
    if len(steps) != graph.poses-1:
        raise ValueError('initialization requires a continuous wheel chain')
    p = np.zeros((graph.poses, 3))
    for i in range(graph.poses-1):
        p[i+1] = compose(p[i], steps[i])
    return p


class Problem:
    """Compile the measurement graph once. No reference poses or landmark map are accepted."""
    def __init__(self, graph, robust_kinds=None):
        validate(graph)
        robust_kinds = set(SIZES if robust_kinds is None else robust_kinds)
        if robust_kinds.difference(SIZES):
            raise ValueError('unknown robust factor kind')
        self.graph = graph
        self.loss_evaluations = 0
        self.dim = 3*(graph.poses-1) + 2*graph.landmarks + 1
        self.groups = []
        self.blocks = []
        robust_blocks = []
        start = 0
        for kind, width in SIZES.items():
            items = [f for f in graph.factors if f.kind == kind]
            if not items:
                continue
            end = start + len(items)*width
            rows = np.arange(start, end).reshape(-1, width)
            self.groups.append((kind, rows, np.array([f.i for f in items]), np.array([f.j for f in items]),
                                np.array([f.z for f in items]), np.array([f.sigma for f in items]),
                                np.array([f.dt for f in items])))
            self.blocks.extend([width] * len(items))
            robust_blocks.extend([kind in robust_kinds] * len(items))
            start = end
        self.rows = start
        self.starts = np.cumsum([0]+self.blocks[:-1])
        self.robust_blocks = np.asarray(robust_blocks, dtype=bool)

    def initial(self):
        poses = wheel_path(self.graph)
        samples = [[] for _ in range(self.graph.landmarks)]
        for f in self.graph.factors:
            if f.kind == 'beacon':
                d, a = f.z
                samples[f.j].append(poses[f.i,:2] + rotate([d, 0.], a + poses[f.i,2]))
        if any(not s for s in samples):
            raise ValueError('unobserved landmark')
        landmarks = np.array([np.median(s, axis=0) for s in samples])
        return np.r_[poses[1:].ravel(), landmarks.ravel(), 0.]

    def unpack(self, x):
        split = 3*(self.graph.poses-1)
        return np.r_[np.zeros(3), x[:split]].reshape(-1,3), x[split:-1].reshape(-1,2), x[-1]

    def linearize(self, x, jac=True):
        poses, marks, bias = self.unpack(x)
        residual = np.empty(self.rows)
        # Work with absolute pose column numbers, then drop the three anchored columns.
        J = np.zeros((self.rows, self.dim+3)) if jac else None
        for kind, rows, i, j, z, sigma, dt in self.groups:
            ri = rows[:,0]
            ca, sa = np.cos(poses[i,2]), np.sin(poses[i,2])
            if kind in ('vo','wheel','loop'):
                body = rotate(poses[j,:2]-poses[i,:2], -poses[i,2])
                err = np.column_stack((body-z[:,:2], wrap(poses[j,2]-poses[i,2]-z[:,2])))
                if jac:
                    rj, rk = rows[:,1], rows[:,2]
                    J[ri,3*j], J[ri,3*j+1], J[rj,3*j], J[rj,3*j+1] = ca, sa, -sa, ca
                    J[ri,3*i], J[ri,3*i+1], J[rj,3*i], J[rj,3*i+1] = -ca, -sa, sa, -ca
                    J[ri,3*i+2], J[rj,3*i+2] = body[:,1], -body[:,0]
                    J[rk,3*i+2], J[rk,3*j+2] = -1., 1.
            elif kind == 'gyro':
                err = wrap(poses[j,2]-poses[i,2]+bias*dt-z[:,0])[:,None]
                if jac:
                    J[ri,3*i+2], J[ri,3*j+2], J[ri,-1] = -1., 1., dt
            else:
                dx, dy = (marks[j]-poses[i,:2]).T
                dist = np.hypot(dx,dy)
                if np.any(dist < 1e-8):
                    raise ValueError('undefined bearing at zero distance')
                err = np.column_stack((dist-z[:,0], wrap(np.arctan2(dy,dx)-poses[i,2]-z[:,1])))
                if jac:
                    col = 3*self.graph.poses+2*j
                    rb = rows[:,1]
                    J[ri,col], J[ri,col+1] = dx/dist, dy/dist
                    J[rb,col], J[rb,col+1] = -dy/dist**2, dx/dist**2
                    J[ri,3*i], J[ri,3*i+1] = -dx/dist, -dy/dist
                    J[rb,3*i], J[rb,3*i+1], J[rb,3*i+2] = dy/dist**2, -dx/dist**2, -1.
            residual[rows] = err/sigma
            if jac:
                J[rows] /= sigma[:,:,None]
        return residual, (J[:,3:] if jac else None)

    def loss(self, r, delta):
        # Huber is applied to the norm of a whole normalized sensor block.
        squared = np.add.reduceat(r*r, self.starts)
        self.loss_evaluations += 1
        if delta is None:
            return .5 * squared.sum(), np.ones_like(squared)
        if not np.isfinite(delta) or delta <= 0:
            raise ValueError('Huber threshold must be positive')
        lengths = np.sqrt(squared)
        tails = self.robust_blocks & (lengths > delta)
        weights = np.ones_like(lengths)
        weights[tails] = delta / lengths[tails]
        cost = np.where(tails, delta*(lengths-.5*delta), .5*squared)
        return cost.sum(), weights

    def objective(self, x, delta):
        return self.loss(self.linearize(x, jac=False)[0], delta)[0]

In [ ]:
%%writefile slam_lab/solver.py
"""Damped Gauss–Newton with one acceptance rule for all damping policies."""
from dataclasses import dataclass
import time
import numpy as np
from .graph import Problem

MIN_LAMBDA, MAX_LAMBDA = 1e-7, 1e5


@dataclass
class Iteration:
    r: np.ndarray
    H: np.ndarray
    g: np.ndarray
    diag: np.ndarray
    cost: float
    features: np.ndarray


def linear_step(problem, x, delta, previous_lambda, rho):
    r, J = problem.linearize(x)
    cost, weights = problem.loss(r, delta)
    weighted = J * np.repeat(weights, problem.blocks)[:,None]
    H, g = weighted.T @ J, weighted.T @ r
    diag = np.maximum(np.diag(H), 1e-8)
    features = np.array([np.log10(1+np.linalg.norm(g,np.inf)), np.log10(1+np.linalg.norm(r)),
                         np.log10(diag.max()/diag.min()), np.mean(weights < 1.),
                         np.log10(previous_lambda), np.clip(rho,-2.,2.)])
    return Iteration(r,H,g,diag,float(cost),features)


def trial(problem, x, state, damping, delta):
    A = state.H.copy()
    A.flat[::len(x)+1] += damping * state.diag
    try:
        dx = np.linalg.solve(A, -state.g)
        candidate = x + dx
        cost = problem.objective(candidate, delta)
        predicted = -state.g@dx - .5*dx@state.H@dx
    except (ValueError, np.linalg.LinAlgError, FloatingPointError):
        return x, np.inf, 0.
    ratio = (state.cost-cost)/predicted if predicted > 0 else 0.
    return candidate, float(cost), float(ratio)


@dataclass
class Estimate:
    x: np.ndarray
    poses: np.ndarray
    landmarks: np.ndarray
    bias: float
    history: list
    solves: int
    rejects: int
    status: str
    seconds: float
    advisor_calls: int = 0
    loss_evaluations: int = 0


def solve(graph, *, delta=2.5, policy='classical', initial_lambda=.003, iterations=24,
          budget=48, x0=None, robust_kinds=None):
    """policy is 'classical', 'fixed', or an object implementing reset() and propose(features)."""
    if not MIN_LAMBDA <= initial_lambda <= MAX_LAMBDA or budget < 1 or iterations < 1:
        raise ValueError('invalid damping or solve budget')
    if isinstance(policy,str) and policy not in ('classical','fixed'):
        raise ValueError('unknown damping policy')
    started = time.perf_counter()
    pb = Problem(graph, robust_kinds)
    x = pb.initial() if x0 is None else np.asarray(x0,dtype=float).copy()
    if x.shape != (pb.dim,) or not np.isfinite(x).all():
        raise ValueError('invalid initial state')
    if not isinstance(policy,str):
        policy.reset()
    previous = current = float(initial_lambda)
    rho = 1.
    history = [float(pb.objective(x, delta))]
    solves = rejects = advisor_calls = 0
    status = 'iteration_limit'
    for _ in range(iterations):
        state = linear_step(pb, x, delta, previous, rho)
        if np.linalg.norm(state.g, np.inf) < 1e-7:
            status = 'small_gradient'
            break
        if isinstance(policy,str):
            damping = initial_lambda if policy == 'fixed' else current
        else:
            advisor_calls += 1
            proposal = float(policy.propose(state.features.copy()))
            damping = np.clip(proposal, MIN_LAMBDA, MAX_LAMBDA) if np.isfinite(proposal) and proposal > 0 else current
        accepted = False
        for _attempt in range(8):
            candidate, cost, new_rho = trial(pb,x,state,damping,delta)
            solves += 1
            if np.isfinite(cost) and cost < state.cost:
                x = candidate
                history.append(cost)
                previous, rho = damping, new_rho
                factor = .35 if rho > .75 else (2. if rho < .25 else 1.)
                current = float(np.clip(damping*factor, MIN_LAMBDA, MAX_LAMBDA))
                accepted = True
                break
            rejects += 1
            if solves == budget or damping >= MAX_LAMBDA:
                break
            damping = min(MAX_LAMBDA, 10*damping)
        if not accepted:
            status = 'budget' if solves >= budget else 'no_descent'
            break
        if state.cost-cost <= 1e-8*(1+state.cost):
            status = 'small_decrease'
            break
        if solves >= budget:
            status = 'budget'
            break
    poses, marks, bias = pb.unpack(x)
    return Estimate(x,poses,marks,float(bias),history,solves,rejects,status,time.perf_counter()-started,
                    advisor_calls,pb.loss_evaluations)

In [ ]:
%%writefile slam_lab/bench.py
"""Synthetic observations, kept apart from the truth used for scoring."""
from dataclasses import dataclass
import numpy as np
from .graph import Factor, Graph, relative, rotate, wrap


@dataclass(frozen=True)
class Reference:
    poses: np.ndarray
    landmarks: np.ndarray
    bias: float


def episode(seed, condition='outliers', length=21):
    if condition not in ('clean','outliers','dropout','false_loop','transfer'):
        raise ValueError('unknown episode condition')
    rng = np.random.default_rng(seed)
    t = np.linspace(0, 2*np.pi, length)
    a, b = rng.uniform(1.7,2.4), rng.uniform(1.,1.5)
    if condition == 'transfer':
        b *= .55
    xy = np.column_stack((a*np.sin(t), b*(1-np.cos(t))))
    yaw = np.arctan2(b*np.sin(t), a*np.cos(t))
    poses = np.column_stack((xy,yaw))
    marks = np.array([[-3.4,.4],[-1.3,4.2],[1.8,4.5],[3.6,.7],[.3,-1.7]]) + rng.normal(0,.15,(5,2))
    bias = rng.uniform(.025,.065)
    factors = []
    def add(kind,i,j,z,sigma):
        factors.append(Factor(kind,i,j,tuple(z),tuple(sigma)))
    for i,p in enumerate(poses):
        if i < length-1:
            rel = relative(p, poses[i+1])
            wheel = rel + rng.normal(0,[.040,.018,.021]) + [.045 + .075*(i%9 < 3),0,.012]
            add('wheel',i,i+1,wheel,[.09,.04,.05])
            visual = rel + rng.normal(0,[.035,.035,.025])
            if rng.random() > .13 and not (condition == 'dropout' and length//4 <= i < 3*length//4):
                add('vo',i,i+1,visual,[.035,.035,.025])
            add('gyro',i,i+1,[wrap(rel[2]+bias+rng.normal(0,.012))],[.012])
        for j in (i % 5,(i+2)%5):
            d = marks[j]-p[:2]
            measure = [np.linalg.norm(d)+rng.normal(0,.065), wrap(np.arctan2(d[1],d[0])-p[2]+rng.normal(0,.028))]
            if condition != 'clean' and (3*i+j)%17 == 0:
                measure[0] += 1.5
                measure[1] = wrap(measure[1]-.43)
            add('beacon',i,j,measure,[.065,.028])
    loop = relative(poses[0],poses[-1]) + rng.normal(0,[.045,.045,.02])
    if condition == 'false_loop':
        loop += [1.6,-.9,.45]
    add('loop',0,length-1,loop,[.055,.055,.025])
    return Graph(length,len(marks),tuple(factors)), Reference(poses,marks,bias)


def score(poses, truth, landmarks=None):
    xy_error = poses[:,:2]-truth.poses[:,:2]
    est = rotate(np.diff(poses[:,:2],axis=0), -poses[:-1,2])
    ref = rotate(np.diff(truth.poses[:,:2],axis=0), -truth.poses[:-1,2])
    def rmse(vec):
        return float(np.sqrt(np.mean(np.sum(vec**2,axis=1))))
    return {'ate_m':rmse(xy_error), 'rpe_m':rmse(est-ref),
            'yaw_rad':float(np.sqrt(np.mean(wrap(poses[:,2]-truth.poses[:,2])**2))),
            'map_m':rmse(landmarks-truth.landmarks) if landmarks is not None else float('nan')}

In [ ]:
%%writefile slam_lab/__init__.py
from .graph import COURSE_HUBER, Factor, Graph, Problem, camera_delta_to_base, compose, relative, validate, wheel_path, wrap
from .solver import Estimate, solve

### Один маршрут
В графе с известными соответствиями одновременно оцениваются 20 свободных поз, пять ориентиров и постоянное смещение гироскопа. Здесь сравниваем колёсную одометрию, WLS и блочный Huber на идентичных наблюдениях.

In [ ]:
import pandas as pd
from slam_lab import solve,wheel_path,Problem
from slam_lab.bench import episode,score
graph,truth=episode(23,'false_loop')
est=solve(graph)
wls=solve(graph,delta=None)
display(pd.DataFrame([score(wheel_path(graph),truth),score(wls.poses,truth,wls.landmarks),
                      score(est.poses,truth,est.landmarks)],index=['Колёса','WLS','Huber']).round(4))

### Повторы, удаление петли и независимый решатель
Повторяем каждый сценарий на восьми seed. В контрольном расчёте SciPy минимизирует обычную сумму квадратов из той же начальной точки, оценивая производные конечными разностями.

In [ ]:
%%writefile study.py
"""Reproduce the benchmark tables and the figures from independently generated episodes."""
from dataclasses import asdict, replace
from pathlib import Path
import json
import platform

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy.optimize import least_squares
from threadpoolctl import threadpool_limits

from slam_lab import COURSE_HUBER, Problem, solve, wheel_path
from slam_lab.bench import episode, score

SEEDS = [23,24,25,26,71,72,73,74]
CONDITIONS = ['clean','outliers','dropout','false_loop']


def interval_counts(graph, seed, scenario):
    rows = [dict(seed=seed, scenario=scenario, interval=i, wheel=0, vo=0, gyro=0, beacon=0)
            for i in range(graph.poses-1)]
    for f in graph.factors:
        if f.kind != 'loop' and f.i < len(rows):
            rows[f.i][f.kind] += 1
    return rows


def run(folder='results'):
    folder = Path(folder)
    folder.mkdir(exist_ok=True, parents=True)
    records = []
    availability = []
    example = None
    with threadpool_limits(limits=1):
        for condition in CONDITIONS:
            for seed in SEEDS:
                graph, truth = episode(seed,condition)
                availability.extend(interval_counts(graph,seed,condition))
                wheel = wheel_path(graph)
                coverage = sum(f.kind == 'vo' for f in graph.factors)/(graph.poses-1)
                records.append(dict(seed=seed, scenario=condition, method='wheel', vo_coverage=coverage,
                                    **score(wheel,truth)))
                for method, delta, omit, kinds in [('WLS',None,False,None),('Huber',2.5,False,None),
                        ('Huber без петли',2.5,True,None),('Huber Colab',2.5,False,COURSE_HUBER)]:
                    active = replace(graph,factors=tuple(f for f in graph.factors if f.kind != 'loop')) if omit else graph
                    sol = solve(active,delta=delta,robust_kinds=kinds)
                    records.append(dict(seed=seed,scenario=condition,method=method,vo_coverage=coverage,
                                        **score(sol.poses,truth,sol.landmarks), bias_rad_s=sol.bias,
                                        bias_error_rad_s=sol.bias-truth.bias, final_cost=sol.history[-1],
                                        accepted=len(sol.history)-1, solves=sol.solves, status=sol.status))
                    if seed == 23 and condition == 'false_loop' and method == 'Huber':
                        example = graph,truth,sol

        graph, truth, sol = example
        pb = Problem(graph)
        start = pb.initial()
        other = least_squares(lambda x:pb.linearize(x,False)[0], start, jac='2-point', method='trf',
                              ftol=1e-12, xtol=1e-12, gtol=1e-10)
        ours = solve(graph,delta=None,iterations=60,budget=100)
        check = {'seed':23,'scenario':'false_loop','own_WLS':ours.history[-1],'scipy_WLS':other.cost,
                 'max_state_diff':float(np.max(np.abs(ours.x-other.x)))}
        assert abs(check['own_WLS']-check['scipy_WLS']) < 1e-4
        (folder/'scipy_check.json').write_text(json.dumps(check,indent=2))
    df = pd.DataFrame(records)
    df.to_csv(folder/'runs.csv',index=False)
    agg = df.groupby(['scenario','method']).agg(ate_median=('ate_m','median'),ate_min=('ate_m','min'),
              ate_max=('ate_m','max'),rpe_median=('rpe_m','median'),map_median=('map_m','median'),
              yaw_median_rad=('yaw_rad','median')).reset_index()
    agg.to_csv(folder/'summary.csv',index=False)
    (folder/'measurements.json').write_text(json.dumps(asdict(graph),indent=2))
    np.savez(folder/'reference.npz',poses=truth.poses,landmarks=truth.landmarks,bias=truth.bias)
    np.savetxt(folder/'trajectory.csv', sol.poses, delimiter=',', header='x_m,y_m,yaw_rad', comments='')
    np.savetxt(folder/'landmarks.csv', sol.landmarks, delimiter=',', header='x_m,y_m', comments='')
    counts = pd.DataFrame(availability)
    counts.to_csv(folder/'factor_availability.csv',index=False)
    focus = counts[(counts.seed==23) & (counts.scenario=='dropout')]
    sensors = ['wheel','vo','gyro','beacon']
    heat, hax = plt.subplots(figsize=(8,2.4),layout='constrained')
    hax.imshow(focus[sensors].to_numpy().T > 0,aspect='auto',interpolation='nearest',
               cmap='Blues',vmin=0,vmax=1)
    hax.set(yticks=range(4),yticklabels=['Колёса','VO','Гироскоп','Ориентиры'],
            xticks=range(0,graph.poses-1,2),xlabel='Начальная поза интервала i → i+1',
            title='Доступность измерений · разрыв VO · seed 23')
    heat.savefig(folder/'availability.png',dpi=150)
    plt.close(heat)
    manifest = dict(seeds=SEEDS,scenarios=CONDITIONS,poses=graph.poses,interval_s=1.0,
                    example={'seed':23,'scenario':'false_loop','estimate':'Huber'},
                    methods=df.method.unique().tolist(), course_huber_kinds=COURSE_HUBER,
                    huber_delta=2.5,initial_lambda=.003,iterations=24,budget=48,
                    python=platform.python_version(),numpy=np.__version__,scipy=scipy.__version__,
                    pandas=pd.__version__,matplotlib=plt.matplotlib.__version__)
    (folder/'run_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2))
    fig,ax = plt.subplots(1,2,figsize=(10,4.1),layout='constrained')
    for path,name,color in [(truth.poses,'Эталон','#222222'),(wheel_path(graph),'Колёса','#bd6c30'),
                             (ours.poses,'WLS','#73879a'),(sol.poses,'Huber','#007c83')]:
        ax[0].plot(*path[:,:2].T,'o-',label=name,color=color,ms=2)
    ax[0].scatter(*truth.landmarks.T,color='black',marker='x',label='Ориентиры, GT')
    ax[0].scatter(*sol.landmarks.T,edgecolor='#007c83',facecolor='none',label='Ориентиры, Huber')
    ax[0].set(aspect='equal',title='Ложная петля · seed 23',xlabel='x, м',ylabel='y, м')
    ax[0].legend(fontsize=8,ncol=2)
    for name,color in [('wheel','#bd6c30'),('WLS','#73879a'),('Huber','#007c83')]:
        part = df[(df.method==name)&(df.scenario=='false_loop')]
        ax[1].plot(range(len(SEEDS)),part.ate_m,'o-',label=name,color=color)
    ax[1].set(xticks=range(len(SEEDS)),xticklabels=SEEDS,xlabel='Seed маршрута',ylabel='ATE, м',title='Разброс по маршрутам')
    ax[1].grid(axis='y',alpha=.25); ax[1].legend()
    fig.savefig(folder/'trajectories.png',dpi=150)
    plt.close(fig)
    return agg


if __name__ == '__main__':
    run()

In [ ]:
from study import run
from IPython.display import Image, display
table=run('results')
display(table.round(4))
display(Image(filename='results/trajectories.png'))
display(Image(filename='results/availability.png'))

### Когда гарантия применима
В линейной задаче с положительно определённой H итерация xₖ₊₁ = xₖ − α(Hxₖ − b) имеет коэффициент сжатия q = maxᵢ |1 − αλᵢ(H)| < 1 при 0 < α < 2/λₘₐₓ. Для нелинейного графа якобиан меняется. На этом стенде решатель проверяет фактическое уменьшение цели на каждом принятом шаге. Тесты ниже охватывают рамки, все столбцы якобиана, ошибочные измерения и линейный пример.

In [ ]:
%%writefile test_graph.py
import unittest
from dataclasses import replace
import numpy as np
from slam_lab import COURSE_HUBER, Problem, camera_delta_to_base, compose, relative, solve, validate, wrap
from slam_lab.bench import episode


class GraphChecks(unittest.TestCase):
    def setUp(self):
        self.graph, self.truth = episode(194)

    def test_frames(self):
        a,b = np.array([3.,-.4,1.8]),np.array([.6,.01,2.1])
        np.testing.assert_allclose(relative(a,compose(a,b)),b,atol=1e-12)
        self.assertAlmostEqual(float(wrap(np.deg2rad(179)-np.deg2rad(-179))),np.deg2rad(-2))

    def test_block_jacobians(self):
        p = Problem(self.graph)
        x = p.initial()
        _, analytic = p.linearize(x)
        h = 1e-6
        numeric = np.stack([(p.linearize(x+h*d,False)[0]-p.linearize(x-h*d,False)[0])/(2*h)
                            for d in np.eye(len(x))],axis=1)
        np.testing.assert_allclose(analytic,numeric,atol=2e-6,rtol=2e-6)

    def test_pnp_in_optical_axes(self):
        # Optical Z points forward, X right, Y down. A lever arm also moves during yaw.
        mounting = np.array([[0,0,1,.21],[-1,0,0,.03],[0,-1,0,.14],[0,0,0,1.]])
        a = .4
        body = np.array([[np.cos(a),-np.sin(a),0,.35],[np.sin(a),np.cos(a),0,-.08],
                         [0,0,1,0],[0,0,0,1]])
        T21 = np.linalg.inv(mounting) @ np.linalg.inv(body) @ mounting
        np.testing.assert_allclose(camera_delta_to_base(T21,mounting),[.35,-.08,a],atol=1e-12)
        with self.assertRaises(ValueError):
            camera_delta_to_base(T21, np.diag([-1,1,1,1]))

    def test_whitening_and_course_scope(self):
        from slam_lab import Factor, Graph
        factors = (Factor('wheel',0,1,(0,0,0),(.5,1,1)),Factor('vo',0,1,(0,0,0),(.5,1,1)))
        pb = Problem(Graph(2,0,factors), COURSE_HUBER)
        r,_ = pb.linearize(np.array([2.,0,0,0]))
        np.testing.assert_allclose(r,[4,0,0,4,0,0])
        cost,weights = pb.loss(r,2.)
        np.testing.assert_allclose(weights,[1,.5])
        self.assertAlmostEqual(cost,14.)

    def test_bad_measurements(self):
        f = self.graph.factors[0]
        for bad in [replace(f,i=-1),replace(f,sigma=(1.,0.,1.)),replace(f,z=(np.nan,1.,2.)),
                    replace(f,j=self.graph.poses),replace(f,z=(2.,)),replace(f,kind='gyro',z=(0.,),sigma=(.1,),dt=0)]:
            with self.subTest(bad=bad), self.assertRaises(ValueError):
                validate(replace(self.graph,factors=(bad,)))
        p=Problem(self.graph)
        x=p.initial()
        obs=next(f for f in self.graph.factors if f.kind=='beacon' and f.i==0)
        x[3*(self.graph.poses-1)+2*obs.j:3*(self.graph.poses-1)+2*obs.j+2]=0
        with self.assertRaisesRegex(ValueError,'zero distance'):
            p.linearize(x)

    def test_anchor_guard_and_bias(self):
        e=solve(self.graph)
        np.testing.assert_equal(e.poses[0],[0.,0.,0.])
        self.assertTrue(np.all(np.diff(e.history)<0))
        self.assertLess(abs(e.bias-self.truth.bias),.045)
        self.assertEqual(len(e.history)-1+e.rejects,e.solves)

    def test_linear_contraction(self):
        B = np.array([[1.,0.],[-1.,1.],[0.,1.]])
        H = B.T@B
        eig = np.linalg.eigvalsh(H)
        alpha = 2./sum(eig)
        q=max(abs(1-alpha*eig))
        x = np.zeros(2); target=np.array([.3,1.1])
        for _ in range(12):
            new=x-alpha*(H@(x-target))
            self.assertLessEqual(np.linalg.norm(new-target),q*np.linalg.norm(x-target)+1e-12)
            x=new


if __name__=='__main__':
    unittest.main()

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "unittest", "test_graph"],
    check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True,
)

### Вывод
При выбросах медианная ATE уменьшилась с 0,392 м у WLS до 0,073 м у Huber. На чистых маршрутах оба метода дали около 0,057 м. Правило Huber учитывает величину конфликта. Для решения о замыкании места нужна дополнительная геометрическая проверка.

На маршруте 23 в сценарии разрыва отсутствуют визуальные измерения на интервалах с 5 по 15 включительно. Колёса, гироскоп и ориентиры остаются доступны. В строке Huber Colab вес ограничивается у VO, петли и ориентиров.